# ECE 6524: Project 3
*Mack Werner*

The majority of the following code comes from the tutorial provided for Part 1 of Project 3 and my version in `Werner_Mack_ECE_6524_Project_3_Tutorial.ipynb`.

## Imports

In [ ]:
import os
import string
import pickle
import numpy as np
import matplotlib.pyplot as plt

from numpy import array
from tensorflow.keras.applications.inception_v3 import (InceptionV3, preprocess_input)
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

from nltk.translate.bleu_score import corpus_bleu

## Paths

In [ ]:
BASE_TEXT_DIR = "/Users/briccs/Desktop/Flickr_8k_Text"
IMAGES_DIR = "/Users/briccs/Desktop/Flickr_8k_Images"
GLOVE_DIR = os.path.join(BASE_TEXT_DIR, "glove")

TOKENS_FILE = os.path.join(BASE_TEXT_DIR, "Flickr_8k.token.txt")
TRAIN_IMAGES_FILE = os.path.join(BASE_TEXT_DIR, "Flickr_8k.trainImages.txt")
TEST_IMAGES_FILE = os.path.join(BASE_TEXT_DIR, "Flickr_8k.testImages.txt")
CLEAN_DESCRIPTIONS_FILE = os.path.join(BASE_TEXT_DIR, "descriptions.txt")
ENCODED_TRAIN_FILE = os.path.join(BASE_TEXT_DIR, "encoded_train_images.pkl")
ENCODED_TEST_FILE = os.path.join(BASE_TEXT_DIR, "encoded_test_images.pkl")
GLOVE_FILE = os.path.join(GLOVE_DIR, "glove.6B.200d.txt")

MODEL_SAVE_DIR = "./model_weights"

## Configuration

In [ ]:
WORD_COUNT_THRESHOLD = 10
EMBEDDING_DIM = 200
EPOCHS = 20
PHOTOS_PER_BATCH = 3

## Text Processing Functions

In [ ]:
def load_raw_descriptions(filename):
    """Load raw caption data from Flickr8k.token.txt."""
    with open(filename, "r") as file:
        doc = file.read()

    descriptions = {}

    for line in doc.split("\n"):
        tokens = line.split()
        if len(tokens) < 2:
            continue

        image_id, image_desc = tokens[0], tokens[1:]
        image_id = image_id.split(".")[0]
        image_desc = " ".join(image_desc)

        if image_id not in descriptions:
            descriptions[image_id] = []

        descriptions[image_id].append(image_desc)

    return descriptions


def clean_descriptions(descriptions):
    """Lowercase, remove punctuation/numbers, and keep alphabetic words only."""
    table = str.maketrans("", "", string.punctuation)

    for image_id, desc_list in descriptions.items():
        for i in range(len(desc_list)):
            desc = desc_list[i].split()
            desc = [word.lower() for word in desc]
            desc = [word.translate(table) for word in desc]
            desc = [word for word in desc if len(word) > 1]
            desc = [word for word in desc if word.isalpha()]
            desc_list[i] = " ".join(desc)


def build_vocabulary(descriptions):
    """Build a set of all unique words in the cleaned descriptions."""
    vocabulary = set()
    for desc_list in descriptions.values():
        for desc in desc_list:
            vocabulary.update(desc.split())
    return vocabulary


def save_descriptions(descriptions, filename):
    """Save cleaned descriptions to disk."""
    lines = []
    for image_id, desc_list in descriptions.items():
        for desc in desc_list:
            lines.append(f"{image_id} {desc}")

    data = "\n".join(lines)
    with open(filename, "w") as file:
        file.write(data)


def load_image_ids(filename):
    """Load image IDs from a Flickr8k split file."""
    with open(filename, "r") as file:
        doc = file.read()

    image_ids = set()
    for line in doc.split("\n"):
        if line.strip():
            identifier = line.split(".")[0]
            image_ids.add(identifier)

    return image_ids


def load_clean_descriptions(filename, image_ids):
    """Load cleaned descriptions and keep only those for selected images."""
    with open(filename, "r") as file:
        doc = file.read()

    filtered_descriptions = {}

    for line in doc.split("\n"):
        tokens = line.split()
        if len(tokens) < 2:
            continue

        image_id, image_desc = tokens[0], tokens[1:]

        if image_id in image_ids:
            if image_id not in filtered_descriptions:
                filtered_descriptions[image_id] = []

            desc = "startseq " + " ".join(image_desc) + " endseq"
            filtered_descriptions[image_id].append(desc)

    return filtered_descriptions


def get_all_captions(descriptions):
    """Flatten description dictionary into a list of captions."""
    all_captions = []
    for desc_list in descriptions.values():
        all_captions.extend(desc_list)
    return all_captions


def build_vocab_from_threshold(captions, threshold):
    """Keep only words that appear at least `threshold` times."""
    word_counts = {}
    for sentence in captions:
        for word in sentence.split():
            word_counts[word] = word_counts.get(word, 0) + 1

    vocab = [word for word, count in word_counts.items() if count >= threshold]
    return vocab


def get_max_caption_length(descriptions):
    """Return the maximum caption length."""
    lines = get_all_captions(descriptions)
    return max(len(line.split()) for line in lines)

## Embeddings

In [ ]:
def load_glove_embeddings(glove_file):
    """Load GloVe embeddings from file."""
    embeddings_index = {}

    with open(glove_file, encoding="utf-8") as f:
        for line in f:
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype="float32")
            embeddings_index[word] = coefs

    return embeddings_index


def create_embedding_matrix(wordtoix, embeddings_index, vocab_size, embedding_dim):
    """Create embedding matrix for the vocabulary."""
    embedding_matrix = np.zeros((vocab_size, embedding_dim))

    for word, i in wordtoix.items():
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

    return embedding_matrix

## Image Feature Extraction

In [ ]:
def build_feature_extractor():
    """Build InceptionV3-based feature extractor."""
    base_model = InceptionV3(weights="imagenet")
    feature_model = Model(base_model.input, base_model.layers[-2].output)
    return feature_model


def extract_image_features_for_ids(images_dir, image_ids, feature_model):
    """Extract CNN features for a selected set of image IDs."""
    features = {}

    for image_id in image_ids:
        img_path = os.path.join(images_dir, image_id + ".jpg")
        if not os.path.exists(img_path):
            continue

        img = image.load_img(img_path, target_size=(299, 299))
        x = image.img_to_array(img)
        x = np.expand_dims(x, axis=0)
        x = preprocess_input(x)

        feature = feature_model.predict(x, verbose=0)
        feature = np.reshape(feature, feature.shape[1])

        features[image_id] = feature

    return features

## Data Generator

In [ ]:
def data_generator(
    descriptions,
    photos,
    wordtoix,
    max_caption_length,
    vocab_size,
    num_photos_per_batch,
):
    """Yield batches for model training."""
    X1, X2, y = [], [], []
    n = 0

    while True:
        for key, desc_list in descriptions.items():
            if key not in photos:
                continue

            n += 1
            photo = photos[key]

            for desc in desc_list:
                seq = [wordtoix[word] for word in desc.split() if word in wordtoix]

                for i in range(1, len(seq)):
                    in_seq, out_seq = seq[:i], seq[i]

                    in_seq = pad_sequences([in_seq], maxlen=max_caption_length)[0]
                    out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]

                    X1.append(photo)
                    X2.append(in_seq)
                    y.append(out_seq)

            if n == num_photos_per_batch:
                yield ((array(X1), array(X2)), array(y))
                X1, X2, y = [], [], []
                n = 0

## Model
Based on Project 3, Figure 1.

In [ ]:
def build_caption_model(vocab_size, max_caption_length, embedding_dim, embedding_matrix):
    """Build the caption generation model."""
    image_input = Input(shape=(2048,))
    image_dropout = Dropout(0.5)(image_input)
    image_dense = Dense(256, activation="relu")(image_dropout)

    text_input = Input(shape=(max_caption_length,))
    text_embedding = Embedding(vocab_size, embedding_dim, mask_zero=True)(text_input)
    text_dropout = Dropout(0.5)(text_embedding)
    text_lstm = LSTM(256)(text_dropout)

    decoder = add([image_dense, text_lstm])
    decoder = Dense(256, activation="relu")(decoder)
    output = Dense(vocab_size, activation="softmax")(decoder)

    model = Model(inputs=[image_input, text_input], outputs=output)

    model.layers[2].set_weights([embedding_matrix])
    model.layers[2].trainable = False

    model.compile(loss="categorical_crossentropy", optimizer="adam")
    return model

## Caption Generation & *BLEU* Score

In [ ]:
def greedy_search(model, photo, wordtoix, ixtoword, max_caption_length):
    """Generate a caption using greedy decoding."""
    in_text = "startseq"

    for _ in range(max_caption_length):
        sequence = [wordtoix[w] for w in in_text.split() if w in wordtoix]
        sequence = pad_sequences([sequence], maxlen=max_caption_length)

        yhat = model.predict([photo, sequence], verbose=0)
        yhat = np.argmax(yhat)

        word = ixtoword.get(yhat)
        if word is None:
            break

        in_text += " " + word

        if word == "endseq":
            break

    final_words = in_text.split()[1:-1]
    return " ".join(final_words)


def evaluate_model_bleu(
    model,
    test_descriptions,
    test_features,
    wordtoix,
    ixtoword,
    max_caption_length,
):
    """Evaluate captioning performance with BLEU scores."""
    actual, predicted = [], []

    for image_id, desc_list in test_descriptions.items():
        if image_id not in test_features:
            continue

        yhat = greedy_search(
            model,
            test_features[image_id].reshape((1, 2048)),
            wordtoix,
            ixtoword,
            max_caption_length,
        )

        references = [desc.split() for desc in desc_list]
        references = [ref[1:-1] for ref in references]  # remove startseq/endseq
        predicted_caption = yhat.split()

        actual.append(references)
        predicted.append(predicted_caption)

    bleu1 = corpus_bleu(actual, predicted, weights=(1.0, 0, 0, 0))
    bleu2 = corpus_bleu(actual, predicted, weights=(0.5, 0.5, 0, 0))
    bleu3 = corpus_bleu(actual, predicted, weights=(0.33, 0.33, 0.33, 0))
    bleu4 = corpus_bleu(actual, predicted, weights=(0.25, 0.25, 0.25, 0.25))

    print("BLEU-1: %f" % bleu1)
    print("BLEU-2: %f" % bleu2)
    print("BLEU-3: %f" % bleu3)
    print("BLEU-4: %f" % bleu4)

## Main Script

In [ ]:
# Load and clean raw descriptions
descriptions = load_raw_descriptions(TOKENS_FILE)
clean_descriptions(descriptions)

vocabulary = build_vocabulary(descriptions)
print("Original Vocabulary Size: %d" % len(vocabulary))

# Save cleaned descriptions
save_descriptions(descriptions, CLEAN_DESCRIPTIONS_FILE)

# Load train/test image IDs
train_ids = load_image_ids(TRAIN_IMAGES_FILE)
test_ids = load_image_ids(TEST_IMAGES_FILE)

print("Training Dataset: %d" % len(train_ids))
print("Test Dataset: %d" % len(test_ids))

# Load cleaned descriptions for each split
train_descriptions = load_clean_descriptions(CLEAN_DESCRIPTIONS_FILE, train_ids)
test_descriptions = load_clean_descriptions(CLEAN_DESCRIPTIONS_FILE, test_ids)

print("Descriptions: train=%d" % len(train_descriptions))
print("Descriptions: test=%d" % len(test_descriptions))

# Build thresholded vocabulary from TRAIN ONLY
all_train_captions = get_all_captions(train_descriptions)
vocab = build_vocab_from_threshold(all_train_captions, WORD_COUNT_THRESHOLD)
print("Preprocessed words: %d" % len(vocab))

# Extract image features
feature_model = build_feature_extractor()

encoding_train = extract_image_features_for_ids(IMAGES_DIR, train_ids, feature_model)
encoding_test = extract_image_features_for_ids(IMAGES_DIR, test_ids, feature_model)

print("Encoded train images:", len(encoding_train))
print("Encoded test images:", len(encoding_test))

# Save extracted features
with open(ENCODED_TRAIN_FILE, "wb") as file:
    pickle.dump(encoding_train, file)

with open(ENCODED_TEST_FILE, "wb") as file:
    pickle.dump(encoding_test, file)

# Create word-index mappings
ixtoword = {}
wordtoix = {}

ix = 1
for word in vocab:
    wordtoix[word] = ix
    ixtoword[ix] = word
    ix += 1

vocab_size = len(ixtoword) + 1
max_caption_length = get_max_caption_length(train_descriptions)
print("Max Description Length: %d" % max_caption_length)

# Load GloVe vectors
embeddings_index = load_glove_embeddings(GLOVE_FILE)
embedding_matrix = create_embedding_matrix(
    wordtoix,
    embeddings_index,
    vocab_size,
    EMBEDDING_DIM,
)

# Build model
model = build_caption_model(
    vocab_size=vocab_size,
    max_caption_length=max_caption_length,
    embedding_dim=EMBEDDING_DIM,
    embedding_matrix=embedding_matrix,
)

# Train model
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

steps = len(train_descriptions) // PHOTOS_PER_BATCH

for epoch in range(EPOCHS):
    generator = data_generator(
        train_descriptions,
        encoding_train,
        wordtoix,
        max_caption_length,
        vocab_size,
        PHOTOS_PER_BATCH,
    )

    model.fit(generator, epochs=1, steps_per_epoch=steps, verbose=1)
    model.save(os.path.join(MODEL_SAVE_DIR, f"model_{epoch}.h5"))

# Test caption generation on one training sample
sample_id = list(encoding_train.keys())[0]
sample_feature = encoding_train[sample_id].reshape((1, 2048))

img_path = os.path.join(IMAGES_DIR, sample_id + ".jpg")
img = plt.imread(img_path)
plt.imshow(img)
plt.axis("off")
plt.show()

print("Greedy:", greedy_search(model, sample_feature, wordtoix, ixtoword, max_caption_length))

# Evaluate on the test set
evaluate_model_bleu(
    model,
    test_descriptions,
    encoding_test,
    wordtoix,
    ixtoword,
    max_caption_length,
)